In [17]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install random

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement random (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for random


In [1]:
%pip install psycopg[binary]

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import psycopg

In [22]:
from psycopg import sql

In [23]:
import random
import string
import pandas as pd


def generate_site_code():
    letters = "".join(random.choices(string.ascii_uppercase, k=3))
    numbers = "".join(random.choices(string.digits, k=3))
    return letters + numbers


def generate_site_data(count):
    data = []
    used_site_codes = set()
    used_coordinates = set()

    while len(data) < count:

        site_code = generate_site_code()
        latitude = round(random.uniform(-90, 90), 2)
        longitude = round(random.uniform(-180, 180), 2)

        # Skip duplicate site codes
        if site_code in used_site_codes:
            continue

        # Skip duplicate coordinates
        if (latitude, longitude) in used_coordinates:
            continue

        used_site_codes.add(site_code)
        used_coordinates.add((latitude, longitude))

        data.append(
            {
                "site_code": site_code,
                "latitude": latitude,
                "longitude": longitude,
            }
        )

    return pd.DataFrame(data)

In [24]:
def create_db_meta(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:
            with conn.cursor() as cur:
                query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
                cur.execute(query)
                print(f"Database '{db_name}' created successfully!")

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [25]:
create_db_meta("meta")

Database 'meta' created successfully!


In [26]:
def create_table_meta():
    try:
        with psycopg.connect(
            dbname="meta",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
        ) as conn:

            with conn.cursor() as cur:
                cur.execute("""
                    CREATE TABLE IF NOT EXISTS metadata (
                        site_name VARCHAR(100) Not NULL,
                        latitude DOUBLE PRECISION Not NULL,
                        longitude DOUBLE PRECISION Not NULL
                    );
                """)

            conn.commit()
            print("metadata table created.")

    except psycopg.Error as e:
        print(e)

In [27]:
create_table_meta()

metadata table created.


In [28]:
def insert_sites():

    sites_df = generate_site_data(10000)

    with psycopg.connect(
        dbname="meta", user="postgres", password="123789", host="localhost", port="5000"
    ) as conn:
        with conn.cursor() as cur:
            for _, row in sites_df.iterrows():
                cur.execute(
                    """
                    INSERT INTO metadata
                    (site_name, latitude, longitude)
                    VALUES (%s, %s, %s)                                                                         
                    """,
                    (row["site_code"], row["latitude"], row["longitude"]),
                )
        conn.commit()

    print("Sites inserted successfully!")

In [29]:
insert_sites()

Sites inserted successfully!


In [30]:
def get_sites():

    with psycopg.connect(
        dbname="meta", user="postgres", password="123789", host="localhost", port="5000"
    ) as conn:

        with conn.cursor() as cur:
            cur.execute(""" 
                SELECT site_name, latitude, longitude
                FROM metadata
            """)

            sites = cur.fetchall()

    return sites

In [ ]:
data = get_sites()

In [33]:
type(data[0])

tuple

In [34]:
len(data)

10000

In [35]:
def create_db_site_weather(db_name):
    try:
        with psycopg.connect(
            dbname="csv_database",
            user="postgres",
            password="123789",
            host="localhost",
            port="5000",
            autocommit=True,
        ) as conn:
            with conn.cursor() as cur:
                query = sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name))
                cur.execute(query)
                print(f"Database '{db_name}' created successfully!")

    except psycopg.Error as e:
        print(f"An error occurred: {e}")

In [36]:
create_db_site_weather("site_weather")

Database 'site_weather' created successfully!


In [ ]:
import time
import datetime
import requests
import psycopg

DB_USER = "postgres"
DB_PASSWORD = "123789"
DB_HOST = "localhost"
DB_PORT = "5000"

DB_NAME = "site_weather"

API_URL = "https://api.open-meteo.com/v1/forecast"

START_DATE = "2026-08-11"
END_DATE = "2026-08-11"

BATCH_SIZE = 30

REQUEST_DELAY = 0.5

RETRY_WAIT = 5

RATE_LIMIT_WAIT = 60

REQUEST_TIMEOUT = 120


with psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            CREATE TABLE IF NOT EXISTS site_weather (
                site_name VARCHAR(100) NOT NULL,
                time_interval TIMESTAMPTZ NOT NULL,
                temperature REAL NOT NULL,
                humidity REAL NOT NULL,
                solar_radiance REAL NOT NULL,

                UNIQUE (site_name, time_interval)
            );
        """)

    conn.commit()


with psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT site_name
            FROM site_weather
            GROUP BY site_name
            HAVING COUNT(*) = 24;
        """)

        completed_sites = {row[0] for row in cur.fetchall()}

pending_sites = []

for row in data:

    site_name = row[0]
    latitude = float(row[1])
    longitude = float(row[2])

    if site_name not in completed_sites:
        pending_sites.append((site_name, latitude, longitude))


print("=" * 60)
print("RESUME CHECK")
print("=" * 60)
print(f"Total sites       : {len(data)}")
print(f"Completed sites   : {len(completed_sites)}")
print(f"Remaining sites   : {len(pending_sites)}")
print("=" * 60)


def fetch_batch(batch):

    # site_names = [site[0] for site in batch]
    latitudes = [str(site[1]) for site in batch]
    longitudes = [str(site[2]) for site in batch]

    params = {
        "latitude": ",".join(latitudes),
        "longitude": ",".join(longitudes),
        "hourly": ",".join(
            [
                "temperature_2m",
                "relative_humidity_2m",
                "direct_radiation",
            ]
        ),
        "start_date": START_DATE,
        "end_date": END_DATE,
    }

    # Keep trying the SAME batch until it succeeds.
    while True:

        try:
            
            # print()
            # print(f"Requesting {len(batch)} sites...")
            # print(f"{site_names[0]} -> {site_names[-1]}")

            response = requests.get(
                API_URL,
                params=params,
                timeout=REQUEST_TIMEOUT,
            )

            if response.status_code == 200:

                result = response.json()

                if isinstance(result, dict):
                    result = [result]

                if not isinstance(result, list):
                    raise ValueError("Invalid API response format.")

                if len(result) != len(batch):
                    raise ValueError(
                        f"Expected {len(batch)} locations, "
                        f"but API returned {len(result)}."
                    )

                print("API SUCCESS")
                return result

            if response.status_code == 429:

                print(
                    f"API rate limit reached. " f"Waiting {RATE_LIMIT_WAIT} seconds..."
                )

                time.sleep(RATE_LIMIT_WAIT)

                print("Retrying SAME batch...")
                continue
            
            
            if response.status_code >= 500:

                print(
                    f"Server error {response.status_code}. "
                    f"Waiting {RETRY_WAIT} seconds..."
                )

                time.sleep(RETRY_WAIT)

                print("Retrying SAME batch...")
                continue


            print(f"API error {response.status_code}. " f"Waiting {RETRY_WAIT} seconds...")

            time.sleep(RETRY_WAIT)

            print("Retrying SAME batch...")

        except (requests.Timeout, requests.ConnectionError) as e:

            print(f"Network error: {e}")
            print(f"Waiting {RETRY_WAIT} seconds...")
            time.sleep(RETRY_WAIT)
            print("Retrying SAME batch...")

        except (requests.RequestException, ValueError) as e:

            print(f"Request/response error: {e}")
            print(f"Waiting {RETRY_WAIT} seconds...")
            time.sleep(RETRY_WAIT)
            print("Retrying SAME batch...")


def extract_site_weather(site_name, weather):

    hourly = weather.get("hourly")

    if not hourly:
        raise ValueError(f"No hourly data for {site_name}")

    times = hourly.get("time", [])
    temperatures = hourly.get("temperature_2m", [])
    humidity = hourly.get("relative_humidity_2m", [])
    radiation = hourly.get("direct_radiation", [])

    # We need exactly 24 records for one day.
    if not (
        len(times) == 24
        and len(temperatures) == 24
        and len(humidity) == 24
        and len(radiation) == 24
    ):
        raise ValueError(
            f"Invalid data for {site_name}: "
            f"time={len(times)}, "
            f"temperature={len(temperatures)}, "
            f"humidity={len(humidity)}, "
            f"radiation={len(radiation)}"
        )

    rows = []

    for i in range(24):

        timestamp = datetime.datetime.fromisoformat(times[i]).replace(
            tzinfo=datetime.timezone.utc
        )

        rows.append(
            (
                site_name,
                timestamp,
                float(temperatures[i]),
                float(humidity[i]),
                float(radiation[i]),
            )
        )

    return rows

def save_batch(cursor, batch, api_results):

    all_rows = []

    for i, site in enumerate(batch):

        site_name = site[0]
        weather = api_results[i]

        rows = extract_site_weather(
            site_name,
            weather,
        )

        all_rows.extend(rows)

    cursor.executemany(
        """
        INSERT INTO site_weather (
            site_name,
            time_interval,
            temperature,
            humidity,
            solar_radiance
        )
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (site_name, time_interval)
        DO NOTHING;
    """,
        all_rows,
    )

    return len(all_rows)



total_pending = len(pending_sites)

if total_pending == 0:

    print("All sites are already complete.")

else:

    total_batches = (total_pending + BATCH_SIZE - 1) // BATCH_SIZE    

    with psycopg.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT,
    ) as conn:

        with conn.cursor() as cur:

            for batch_number, start in enumerate(
                range(0, total_pending, BATCH_SIZE),
                start=1,
            ):

                batch = pending_sites[start : start + BATCH_SIZE]

                # print()
                # print("=" * 60)
                # print(f"BATCH {batch_number}/{total_batches}")
                # print(f"Sites: {start + 1} - " f"{start + len(batch)}")
                # print("=" * 60)
                
                # API

                api_results = fetch_batch(batch)

                # DATABASE

                rows_inserted = save_batch(
                    cur,
                    batch,
                    api_results,
                )
                
                conn.commit()

                total_rows += rows_inserted

                print(
                    f"Batch complete | "
                    f"Sites: {len(batch)} | "
                    f"Rows: {rows_inserted} | "
                    f"Total rows: {total_rows}"
                )

                time.sleep(REQUEST_DELAY)
                
                
with psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT COUNT(DISTINCT site_name)
            FROM site_weather;
        """)

        actual_sites = cur.fetchone()[0]

        cur.execute("""
            SELECT COUNT(*)
            FROM site_weather;
        """)

        actual_rows = cur.fetchone()[0]


expected_sites = len(data)
expected_rows = expected_sites * 24

print()
print("=" * 60)
print("FINAL VERIFICATION")
print("=" * 60)
print(f"Expected sites : {expected_sites}")
print(f"Actual sites   : {actual_sites}")
print(f"Expected rows  : {expected_rows}")
print(f"Actual rows    : {actual_rows}")
print("=" * 60)

if actual_sites == expected_sites and actual_rows == expected_rows:
    print("SUCCESS: All sites and all hourly records are complete.")
else:
    print("NOT COMPLETE YET.")
    print(f"Missing rows: {expected_rows - actual_rows}")

RESUME CHECK
Total sites       : 10000
Completed sites   : 0
Remaining sites   : 10000

BATCH 1/334
Sites: 1 - 30

Requesting 30 sites...
XRV562 -> JHL532
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 240720

BATCH 2/334
Sites: 31 - 60

Requesting 30 sites...
ACO651 -> SGM709
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 241440

BATCH 3/334
Sites: 61 - 90

Requesting 30 sites...
MWT237 -> HKK346
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 242160

BATCH 4/334
Sites: 91 - 120

Requesting 30 sites...
LIS206 -> LJY221
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 242880

BATCH 5/334
Sites: 121 - 150

Requesting 30 sites...
FQT834 -> TPX753
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 243600

BATCH 6/334
Sites: 151 - 180

Requesting 30 sites...
HDE170 -> HSA746
API SUCCESS
Batch complete | Sites: 30 | Rows: 720 | Total rows: 244320

BATCH 7/334
Sites: 181 - 210

Requesting 30 sites...
UEO578 ->